# Temporal OOD Experiments

**Goal**: Evaluate CAGP under temporal-like distribution shift.

**Approach**: Use FB15k-237 with frequency-based split to simulate temporal dynamics:
- High-frequency entities = "established" (like older entities in temporal KG)
- Low-frequency entities = "emerging" (like newer entities)

This simulates the key temporal challenge: detecting when familiar entities appear in unfamiliar contexts.

In [ ]:
# Setup
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import pandas as pd
from collections import defaultdict, Counter
from sklearn.metrics import roc_auc_score
import json
import os
import random

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

## 1. Load FB15k-237

In [ ]:
# Download FB15k-237 if not present
# Handle both local (relative to notebooks/) and Colab environments
import os

# Determine correct data path
if os.path.exists('../data/raw/fb15k-237/train.txt'):
    DATA_DIR = '../data/raw/fb15k-237'
else:
    DATA_DIR = 'data/fb15k-237'
    os.makedirs(DATA_DIR, exist_ok=True)

# Check if data exists, download if needed
if not os.path.exists(f'{DATA_DIR}/train.txt'):
    print("Downloading FB15k-237...")
    import urllib.request
    
    # Try multiple sources (some may be unavailable)
    sources = [
        "https://raw.githubusercontent.com/DeepGraphLearning/pLogicNet/master/data/FB15k-237/",
        "https://raw.githubusercontent.com/thunlp/OpenKE/OpenKE-PyTorch/benchmarks/FB15K237/",
    ]
    
    downloaded = False
    for url in sources:
        try:
            print(f"  Trying {url}...")
            for fname in ['train.txt', 'valid.txt', 'test.txt']:
                urllib.request.urlretrieve(f"{url}{fname}", f"{DATA_DIR}/{fname}")
            print("Download complete!")
            downloaded = True
            break
        except Exception as e:
            print(f"  Failed: {e}")
            continue
    
    if not downloaded:
        raise RuntimeError("Could not download FB15k-237 from any source. Please download manually.")
else:
    print(f"Data already exists at {DATA_DIR}")

def load_triples(filepath):
    triples = []
    with open(filepath) as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) == 3:
                triples.append(tuple(parts))
    return triples

train_raw = load_triples(f'{DATA_DIR}/train.txt')
valid_raw = load_triples(f'{DATA_DIR}/valid.txt')
test_raw = load_triples(f'{DATA_DIR}/test.txt')

all_triples = train_raw + valid_raw + test_raw

# Build entity and relation maps
entities = sorted(set([t[0] for t in all_triples] + [t[2] for t in all_triples]))
relations = sorted(set([t[1] for t in all_triples]))

entity2id = {e: i for i, e in enumerate(entities)}
relation2id = {r: i for i, r in enumerate(relations)}
id2entity = {i: e for e, i in entity2id.items()}

n_entities = len(entity2id)
n_relations = len(relation2id)

print(f"Entities: {n_entities}")
print(f"Relations: {n_relations}")
print(f"Train triples: {len(train_raw)}")

## 2. Compute Entity Frequencies (Proxy for "Age")

In [ ]:
# Count entity frequency in training data
entity_freq = Counter()
for h, r, t in train_raw:
    entity_freq[entity2id[h]] += 1
    entity_freq[entity2id[t]] += 1

# Frequency statistics
freqs = [entity_freq[i] for i in range(n_entities)]
print(f"Entity frequency stats:")
print(f"  Min: {min(freqs)}")
print(f"  Max: {max(freqs)}")
print(f"  Median: {np.median(freqs):.0f}")
print(f"  Mean: {np.mean(freqs):.1f}")

# Categorize entities by frequency
freq_threshold_low = np.percentile(freqs, 25)   # Bottom 25% = "new/rare"
freq_threshold_high = np.percentile(freqs, 75)  # Top 25% = "established"

print(f"\nFrequency thresholds:")
print(f"  Low (25th percentile): {freq_threshold_low}")
print(f"  High (75th percentile): {freq_threshold_high}")

## 3. Create Temporal-Like Split

**Protocol**:
- Train on all training triples (simulates "historical" data)
- Test ID: triples where both entities are high-frequency ("established")
- Test OOD-NewEntity: triples with at least one low-frequency entity ("emerging")
- Test OOD-NewPair: triples with established entities but rare relation pattern

In [ ]:
# Convert to ID format
train_triples = [(entity2id[h], relation2id[r], entity2id[t]) for h, r, t in train_raw]
test_triples = [(entity2id[h], relation2id[r], entity2id[t]) for h, r, t in test_raw]

# Build coverage matrix from training
coverage = np.zeros((n_entities, n_relations), dtype=np.float32)
for h, r, t in train_triples:
    coverage[h, r] = 1
    coverage[t, r] = 1

# Entity-relation pairs in training
train_pairs = set()
for h, r, t in train_triples:
    train_pairs.add((h, r))
    train_pairs.add((t, r))

print(f"Coverage matrix: {coverage.shape}")
print(f"Coverage density: {coverage.mean():.4f}")

In [ ]:
def categorize_triple(h, r, t, entity_freq, coverage, freq_threshold_low, freq_threshold_high):
    """Categorize test triple for temporal-like OOD."""
    h_freq = entity_freq[h]
    t_freq = entity_freq[t]
    
    h_covered = coverage[h, r] == 1
    t_covered = coverage[t, r] == 1
    
    # New entity: at least one entity is low-frequency ("emerging")
    if h_freq <= freq_threshold_low or t_freq <= freq_threshold_low:
        return 'new_entity'
    
    # New pair: established entities but unseen entity-relation combination
    if not h_covered or not t_covered:
        return 'new_pair'
    
    # ID: both entities established and both covered for this relation
    return 'id'

# Categorize test triples
test_categories = []
for h, r, t in test_triples:
    cat = categorize_triple(h, r, t, entity_freq, coverage, freq_threshold_low, freq_threshold_high)
    test_categories.append(cat)

# Create DataFrame
test_df = pd.DataFrame({
    'head': [t[0] for t in test_triples],
    'relation': [t[1] for t in test_triples],
    'tail': [t[2] for t in test_triples],
    'category': test_categories
})

print("Test triple categories:")
print(test_df['category'].value_counts())

## 4. Define GP-KGE Model

In [ ]:
class GPKGE(torch.nn.Module):
    def __init__(self, n_entities, n_relations, dim=100):
        super().__init__()
        self.entity_mean = torch.nn.Embedding(n_entities, dim)
        self.entity_logvar = torch.nn.Embedding(n_entities, dim)
        self.relation_emb = torch.nn.Embedding(n_relations, dim)
        
        torch.nn.init.xavier_uniform_(self.entity_mean.weight)
        torch.nn.init.constant_(self.entity_logvar.weight, -2.0)
        torch.nn.init.xavier_uniform_(self.relation_emb.weight)
    
    def forward(self, heads, relations, tails):
        h_mean = self.entity_mean(heads)
        t_mean = self.entity_mean(tails)
        r = self.relation_emb(relations)
        score = (h_mean * r * t_mean).sum(dim=-1)
        return score
    
    def get_variance(self, entities):
        logvar = self.entity_logvar(entities)
        return torch.exp(logvar).mean(dim=-1)
    
    def get_gp_uncertainty(self, heads, tails):
        h_var = self.get_variance(heads)
        t_var = self.get_variance(tails)
        return (h_var + t_var) / 2

In [ ]:
def train_gpkge(model, train_triples, n_entities, epochs=30, batch_size=2048, lr=0.001, device='cuda'):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = torch.nn.BCEWithLogitsLoss()
    
    heads = torch.tensor([t[0] for t in train_triples])
    relations = torch.tensor([t[1] for t in train_triples])
    tails = torch.tensor([t[2] for t in train_triples])
    n_triples = len(heads)
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        perm = torch.randperm(n_triples)
        
        for i in range(0, n_triples, batch_size):
            idx = perm[i:i+batch_size]
            h = heads[idx].to(device)
            r = relations[idx].to(device)
            t = tails[idx].to(device)
            
            pos_scores = model(h, r, t)
            neg_t = torch.randint(0, n_entities, (len(idx),)).to(device)
            neg_scores = model(h, r, neg_t)
            
            scores = torch.cat([pos_scores, neg_scores])
            labels = torch.cat([torch.ones_like(pos_scores), torch.zeros_like(neg_scores)])
            loss = criterion(scores, labels)
            
            kl = 0.5 * (model.entity_logvar.weight.exp() - model.entity_logvar.weight - 1).mean()
            loss = loss + 0.01 * kl
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")
    
    return model

## 5. Train Model

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

model = GPKGE(n_entities, n_relations, dim=100)
model = train_gpkge(model, train_triples, n_entities, epochs=30, device=device)

## 6. Evaluate Temporal OOD

In [ ]:
def evaluate_temporal_ood(model, coverage, test_df, device='cuda'):
    model.eval()
    
    id_df = test_df[test_df['category'] == 'id']
    ood_df = test_df[test_df['category'].isin(['new_entity', 'new_pair'])]
    
    print(f"ID samples: {len(id_df)}")
    print(f"OOD samples: {len(ood_df)}")
    print(f"  - new_entity: {len(test_df[test_df['category']=='new_entity'])}")
    print(f"  - new_pair: {len(test_df[test_df['category']=='new_pair'])}")
    
    def get_uncertainties(df):
        heads = torch.tensor(df['head'].values).to(device)
        relations = torch.tensor(df['relation'].values).to(device)
        tails = torch.tensor(df['tail'].values).to(device)
        
        with torch.no_grad():
            gp_unc = model.get_gp_uncertainty(heads, tails).cpu().numpy()
        
        h_np = heads.cpu().numpy()
        r_np = relations.cpu().numpy()
        t_np = tails.cpu().numpy()
        cov_unc = 2 - coverage[h_np, r_np] - coverage[t_np, r_np]
        
        return gp_unc, cov_unc
    
    id_gp, id_cov = get_uncertainties(id_df)
    ood_gp, ood_cov = get_uncertainties(ood_df)
    
    labels = np.concatenate([np.zeros(len(id_gp)), np.ones(len(ood_gp))])
    
    gp_scores = np.concatenate([id_gp, ood_gp])
    cov_scores = np.concatenate([id_cov, ood_cov])
    
    auroc_gp = roc_auc_score(labels, gp_scores)
    auroc_cov = roc_auc_score(labels, cov_scores)
    
    gp_norm = gp_scores * cov_scores.mean() / (gp_scores.mean() + 1e-8)
    cagp_scores = 0.5 * gp_norm + 0.5 * cov_scores
    auroc_cagp = roc_auc_score(labels, cagp_scores)
    
    return {
        'GP-only': auroc_gp,
        'Coverage-only': auroc_cov,
        'CAGP': auroc_cagp,
        'synergy': auroc_cagp - max(auroc_gp, auroc_cov)
    }

results = evaluate_temporal_ood(model, coverage, test_df, device=device)

print("\n" + "="*50)
print("TEMPORAL-LIKE OOD RESULTS (FB15k-237)")
print("="*50)
for method, auroc in results.items():
    print(f"{method}: {auroc:.4f}")

## 7. Breakdown by OOD Type

In [ ]:
def evaluate_by_ood_type(model, coverage, test_df, device='cuda'):
    model.eval()
    
    id_df = test_df[test_df['category'] == 'id']
    new_entity_df = test_df[test_df['category'] == 'new_entity']
    new_pair_df = test_df[test_df['category'] == 'new_pair']
    
    def get_uncertainties(df):
        if len(df) == 0:
            return np.array([]), np.array([])
        heads = torch.tensor(df['head'].values).to(device)
        relations = torch.tensor(df['relation'].values).to(device)
        tails = torch.tensor(df['tail'].values).to(device)
        
        with torch.no_grad():
            gp_unc = model.get_gp_uncertainty(heads, tails).cpu().numpy()
        
        h_np, r_np, t_np = heads.cpu().numpy(), relations.cpu().numpy(), tails.cpu().numpy()
        cov_unc = 2 - coverage[h_np, r_np] - coverage[t_np, r_np]
        return gp_unc, cov_unc
    
    id_gp, id_cov = get_uncertainties(id_df)
    ne_gp, ne_cov = get_uncertainties(new_entity_df)
    np_gp, np_cov = get_uncertainties(new_pair_df)
    
    results = {}
    
    # ID vs New Entity
    if len(ne_gp) > 0 and len(id_gp) > 0:
        labels = np.concatenate([np.zeros(len(id_gp)), np.ones(len(ne_gp))])
        gp_scores = np.concatenate([id_gp, ne_gp])
        cov_scores = np.concatenate([id_cov, ne_cov])
        gp_norm = gp_scores * cov_scores.mean() / (gp_scores.mean() + 1e-8)
        cagp_scores = 0.5 * gp_norm + 0.5 * cov_scores
        
        results['new_entity'] = {
            'GP-only': roc_auc_score(labels, gp_scores),
            'Coverage-only': roc_auc_score(labels, cov_scores),
            'CAGP': roc_auc_score(labels, cagp_scores),
            'n_samples': len(ne_gp)
        }
    
    # ID vs New Pair
    if len(np_gp) > 0 and len(id_gp) > 0:
        labels = np.concatenate([np.zeros(len(id_gp)), np.ones(len(np_gp))])
        gp_scores = np.concatenate([id_gp, np_gp])
        cov_scores = np.concatenate([id_cov, np_cov])
        gp_norm = gp_scores * cov_scores.mean() / (gp_scores.mean() + 1e-8)
        cagp_scores = 0.5 * gp_norm + 0.5 * cov_scores
        
        results['new_pair'] = {
            'GP-only': roc_auc_score(labels, gp_scores),
            'Coverage-only': roc_auc_score(labels, cov_scores),
            'CAGP': roc_auc_score(labels, cagp_scores),
            'n_samples': len(np_gp)
        }
    
    return results

breakdown = evaluate_by_ood_type(model, coverage, test_df, device=device)

print("\n" + "="*50)
print("BREAKDOWN BY OOD TYPE")
print("="*50)
for ood_type, metrics in breakdown.items():
    print(f"\n{ood_type.upper()} (n={metrics['n_samples']}):")
    print(f"  GP-only:       {metrics['GP-only']:.4f}")
    print(f"  Coverage-only: {metrics['Coverage-only']:.4f}")
    print(f"  CAGP:          {metrics['CAGP']:.4f}")

## 8. Save Results

In [ ]:
os.makedirs('outputs', exist_ok=True)

output = {
    'dataset': 'FB15k-237',
    'split_type': 'frequency-based temporal simulation',
    'description': 'Low-frequency entities treated as emerging/new, high-frequency as established',
    'overall': results,
    'breakdown': breakdown,
    'metadata': {
        'n_entities': n_entities,
        'n_relations': n_relations,
        'train_triples': len(train_triples),
        'test_id': len(test_df[test_df['category'] == 'id']),
        'test_ood_new_entity': len(test_df[test_df['category'] == 'new_entity']),
        'test_ood_new_pair': len(test_df[test_df['category'] == 'new_pair']),
        'freq_threshold_low': float(freq_threshold_low),
        'freq_threshold_high': float(freq_threshold_high),
    }
}

with open('outputs/temporal_ood_results.json', 'w') as f:
    json.dump(output, f, indent=2)

print("Results saved to outputs/temporal_ood_results.json")

## 9. Key Insights

**Expected Results**:
1. **New Entity OOD**: GP should excel (high variance for rare/emerging entities)
2. **New Pair OOD**: Coverage should excel (established entity in new relational context)
3. **CAGP**: Should capture both failure modes

**Why This Simulates Temporal OOD**:
- In real temporal KGs, new entities appear over time with few initial observations
- Entity frequency correlates with "age" in the knowledge graph
- Low-frequency entities behave like newly emerged entities
- This captures the key challenge: detecting unfamiliar patterns for familiar vs unfamiliar entities